# INFO
* 전방 도킹가이드 자기장센서 계측 Log를 통해, classification
* initialValueLog: 초기 Offset 값 계측 로그
* logData: 자석이 위치한 상태에서의 계측 로그


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyts.image import GramianAngularField
from keras.models import load_model

In [2]:
########## Global Variables ##########
sensorIdx = [4, 9, 14, 20, 25, 30]
sensorName = [f"s{i}" for i in np.arange(1, 7, 1)]
gaf = GramianAngularField(method= "difference")
colName = "col"
skipNumber = 300
useRow = 4000

In [3]:
initialValueLog = pd.read_table("../data/test0319/initialValue.log", names= [colName])
initialValues = []
for i in range(initialValueLog.shape[0]):
    if len(initialValueLog.iloc[i, 0]) > 41 or len(initialValueLog.iloc[i, 0]) < 39:
        pass
    else:
        initialValues.append(initialValueLog.iloc[i, 0])
initialValues = np.array(initialValues)

initialSensors = []
for i in sensorIdx:
    initialSensors.append([s[i : i + 5] for s in initialValues])
initialSensors = pd.DataFrame(np.array(initialSensors, dtype= np.float32).T, columns= sensorName)

# meanInitialSensors = pd.DataFrame(initialSensors.mean(axis= 0), columns= ["init"])
meanInitialSensors = initialSensors.mean(axis= 0).to_numpy()

In [87]:
logData = pd.read_table("../data/test0319/class25.log", names= [colName], skiprows= skipNumber, nrows= useRow)
data = []
for i in range(logData.shape[0]):
    if len(logData.iloc[i, 0]) > 41 or len(logData.iloc[i, 0]) < 39:
        pass
    else:
        data.append(logData.iloc[i, 0])
data = np.array(data)

sensorData = []
for i in sensorIdx:
    sensorData.append([d[i : i + 5] for d in data])
sensorData = np.array(sensorData, dtype= np.float32)
sensorData.shape

(6, 3999)

In [88]:
calibrated = [sensorData[i, :] - meanInitialSensors[i] for i in range(6)]
calibrated = np.array(calibrated)
calibrated.shape

(6, 3999)

In [89]:
splited = []
# for i in range(int(calibrated.shape[0] / 16)):
#     splited.append(calibrated[i * 16 : 16 + (i * 16), :])
# splited = np.array(splited)
# splited.shape

for i in range(int(calibrated.shape[1] / 16)):
    splited.append(calibrated[:, i * 16 : 16 + (16 * i)])
splited = np.array(splited)
splited.shape

(249, 6, 16)

In [90]:
s1 = splited[:, 0, :]
s2 = splited[:, 1, :]
s3 = splited[:, 2, :]
s4 = splited[:, 3, :]
s5 = splited[:, 4, :]
s6 = splited[:, 5, :]

In [84]:
s1.shape

(249, 16)

In [91]:
def encoding(s):
    encode = []
    for i in range(s.shape[0]):
        d = s[i, :].reshape(1, -1)
        e = gaf.fit_transform(d).reshape(16, 16, 1)
        encode.append(e)
    encode = np.array(encode)
    return encode

In [92]:
s1Encode = encoding(s1)
print(s1Encode.shape)
s2Encode = encoding(s2)
s3Encode = encoding(s3)
s4Encode = encoding(s4)
s5Encode = encoding(s5)
s6Encode = encoding(s6)

encode = [s1Encode, s2Encode, s3Encode, s4Encode, s5Encode, s6Encode]
encode = np.concatenate(encode, axis= 3)        # (249, 6, 16, 16)


(249, 16, 16, 1)


In [73]:
encode.shape

(249, 16, 16, 6)

In [ ]:
encode = []
for i in range(splited.shape[0]):                   # 249개 데이터
    res = []
    for j in range(splited.shape[2]):
        s = splited[i, :, j].reshape(-1, 1)
        e = gaf.fit_transform(s.T).reshape(16, 16)
        res.append(e)
    res = np.array(res)
    encode.append(res)
encode = np.array(encode)
encode.shape

In [76]:
encode.shape

(249, 16, 16, 6)

In [93]:
re = encode.reshape(encode.shape[0], 1, encode.shape[1], encode.shape[2], encode.shape[3])

In [94]:
trainModel = load_model("../data/Test0919_6Channel_Epoch_100.h5")

predictResult = []
for i in range(re.shape[0]):
    predictClass = np.argmax(trainModel.predict(re[i]), axis= 1)
    predictResult.append(predictClass)

2025-03-19 17:38:57.346058: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


1/1 [==============================] - 0s 19ms/step


In [95]:
print(np.unique(predictResult, return_counts= True))

(array([ 2,  9, 11, 15, 19, 21, 26]), array([  3,   2,   4,   1,   1,  67, 171]))


# Data description
1. initialValue: 전방 초기값
2. class21: class 21정답 데이터
3. class25: class 25 위치 데이터
4. class25_2: class 25 - 3번 센서 근처